# Sentiment Analysis on Amazon Reviews — narrative walkthroughThis notebook is the **story**, not the implementation. Every function it calls lives in a testedpackage at the repo root, so the notebook cannot drift from the code that produced the publishednumbers — if the library changes and this notebook breaks, `scripts/run_notebook.py` fails and CInotices.- The original coursework notebook is preserved unmodified at  [`sentiment_analysis_roberta_ORIGINAL.ipynb`](sentiment_analysis_roberta_ORIGINAL.ipynb). It was  published to Kaggle with **all 28 code cells saved at `outputs: []`**, so none of its metrics  existed anywhere until this repo re-ran both models locally.- This walkthrough runs `cfg/dev.yaml` — a deliberately small config so it executes in minutes and  its outputs are unambiguous about which configuration produced them.- **The published headline numbers come from `cfg/small.yaml`, not from this notebook.** See  [`../reports/RESULTS.md`](../reports/RESULTS.md).

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd() if (Path.cwd() / "train.py").exists() else Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

from cfg.schema import load_config

cfg = load_config(REPO_ROOT / "cfg" / "dev.yaml")
print(cfg.NAME, "|", cfg.DESCRIPTION.strip())
print(f"train={cfg.DATA.N_TRAIN}  test={cfg.DATA.N_TEST}  "
      f"val_fraction={cfg.DATA.VAL_FRACTION}  epochs={cfg.MODEL.EPOCHS}  max_len={cfg.MODEL.MAX_LEN}")

## 1. DataThe loader normalises two upstream layouts onto one schema — `label` ∈ {0,1} / `title` / `text`.HuggingFace uses `label` ∈ {0,1}; the Kaggle CSV is headerless with `polarity` ∈ {1,2}. Getting thatremap backwards inverts every label and yields roughly `1 - accuracy`.This cell falls back to the **committed 1,000-row sample** if the full parquet has not been fetched,so the notebook runs on a fresh clone.

In [ ]:
from datasets.loading import class_balance, load_any

train_path = REPO_ROOT / cfg.DATA.TRAIN_PATH
test_path = REPO_ROOT / cfg.DATA.TEST_PATH
if not train_path.exists():
    print("data/raw not fetched — falling back to the committed sample (run `make data` for the full subset)")
    train_path = REPO_ROOT / "data" / "sample" / "reviews_sample.csv"
    test_path = REPO_ROOT / "data" / "sample" / "reviews_sample_test.csv"

train_frame = load_any(train_path, cfg.DATA.ROWS_READ_TRAIN)
test_frame = load_any(test_path, cfg.DATA.ROWS_READ_TEST)

print("train rows read:", len(train_frame), class_balance(train_frame))
print("test  rows read:", len(test_frame), class_balance(test_frame))
train_frame.head(3)

## 2. SplitsThe original notebook had exactly two splits and no validation set, which is why its choice of5 epochs was unjustifiable — there was nothing to early-stop on, and selecting an epoch by testaccuracy would have been leakage. Here a stratified validation split is carved out of train, and thetest split comes from a physically different upstream file so overlap is structurally impossible.

In [ ]:
from datasets.splits import combined_text, make_splits

n_train = min(cfg.DATA.N_TRAIN, len(train_frame))
n_test = min(cfg.DATA.N_TEST, len(test_frame))

splits = make_splits(
    train_frame, test_frame,
    n_train=n_train, n_test=n_test,
    val_fraction=cfg.DATA.VAL_FRACTION, seed=cfg.SEED,
)
print(splits.sizes())
print("train ∩ val indices:", set(splits.train_index) & set(splits.val_index))
combined_text(splits.test).iloc[0][:200]

## 3. The control is a real opponentAmazon polarity is close to linearly separable in bag-of-words space. A well-configured linear modelis a serious baseline, not a straw man — and the interesting question is how close it gets.**The finding lives here.** The source notebook's preprocessing chain deletes negation twice over:`^\\w+$` destroys `n't` before the stopword filter runs, and NLTK's English stopword list contains`not`, `no` and `nor`. With unigrams only, `"not good"` and `"good"` become the same feature vector —on the one task where negation decides the label.

In [ ]:
import numpy as np

from datasets.text_preprocess import build_vectorizer, preprocess_text

for chain, flags in [
    ("notebook chain     ", dict(alphanumeric_only=True, remove_stopwords=True, stem=True)),
    ("negation preserved ", dict(alphanumeric_only=False, remove_stopwords=False, stem=False)),
]:
    a = preprocess_text("this is good", lowercase=True, **flags)
    b = preprocess_text("this is not good", lowercase=True, **flags)
    vec = build_vectorizer(ngram_range=(1, 1))
    m = vec.fit_transform([a, b, "unrelated text about shipping", "unrelated text about packaging"]).toarray()
    print(f"{chain} 'good' -> {a!r:22}  'not good' -> {b!r:26}  identical vectors: {np.allclose(m[0], m[1])}")

In [ ]:
from metrics.classification import classification_metrics, report_text
from metrics.significance import accuracy_interval
from models.registry import create_model

x_train, y_train = list(combined_text(splits.train)), [int(v) for v in splits.train["label"]]
x_test, y_test = list(combined_text(splits.test)), [int(v) for v in splits.test["label"]]

baseline = create_model(
    "tfidf_logreg", seed=cfg.SEED, C=cfg.BASELINE.C, max_iter=cfg.BASELINE.MAX_ITER,
    lowercase=cfg.PREPROCESSING.LOWERCASE,
    alphanumeric_only=cfg.PREPROCESSING.ALPHANUMERIC_ONLY,
    remove_stopwords=cfg.PREPROCESSING.REMOVE_STOPWORDS,
    stem=cfg.PREPROCESSING.STEM,
    ngram_range=tuple(cfg.PREPROCESSING.NGRAM_RANGE),
).fit(x_train, y_train)

base_pred = baseline.predict(x_test)
base_ci = accuracy_interval(y_test, base_pred)
print("TF-IDF + LogReg  accuracy", base_ci.format(), f"(±{base_ci.pp_halfwidth():.1f} pp)")
print(report_text(y_test, base_pred))

## 4. Fine-tuning `roberta-base`Two things here differ from the original notebook and both matter:- `attn_implementation="eager"` is passed to the **model**, not the config. On the config it is a  silent no-op, and on `transformers` 5.x the default `sdpa` returns an **empty** attentions tuple —  the attention figures below would have had nothing to plot.- The epoch is selected on **validation** loss and the test set is scored exactly once.The run is bounded by `RUNTIME.WALL_CLOCK_CAP_MIN`; nothing in this repo runs unbounded.

In [ ]:
from utils.device import power_mode_label, resolve_device
from utils.seeding import set_seed

set_seed(cfg.SEED)
device = resolve_device(cfg.RUNTIME.DEVICE)
print("device:", device, "|", power_mode_label())

roberta = create_model(
    "roberta", pretrained=cfg.MODEL.PRETRAINED, num_labels=cfg.MODEL.NUM_LABELS,
    max_len=cfg.MODEL.MAX_LEN, batch_size=cfg.MODEL.BATCH_SIZE, epochs=cfg.MODEL.EPOCHS,
    lr=cfg.MODEL.LR, seed=cfg.SEED, device=device,
    wall_clock_cap_min=cfg.RUNTIME.WALL_CLOCK_CAP_MIN,
    random_weight_layers=cfg.MODEL.RANDOM_WEIGHT_LAYERS,
)
print("attention implementation:", roberta.model.config._attn_implementation)

In [ ]:
x_val, y_val = list(combined_text(splits.val)), [int(v) for v in splits.val["label"]]
roberta.fit_with_validation(x_train, y_train, x_val, y_val)

for h in roberta.train_report["history"]:
    print(f"epoch {int(h['epoch'])}  train_loss {h['train_loss']:.4f}  "
          f"val_loss {h['val_loss']:.4f}  val_acc {h['val_accuracy']:.4f}  {h['epoch_seconds']:.0f}s")
print("selected epoch:", roberta.train_report["selected_epoch"],
      "|", roberta.train_report["selection_criterion"])
print("wall-clock capped:", roberta.train_report["wall_clock_capped"])

In [ ]:
rob_pred = roberta.predict(x_test)
rob_ci = accuracy_interval(y_test, rob_pred)
print("RoBERTa  accuracy", rob_ci.format(), f"(±{rob_ci.pp_halfwidth():.1f} pp)")
print(report_text(y_test, rob_pred))
print("truncation at max_len", cfg.MODEL.MAX_LEN, ":", roberta.evaluate_truncation(x_test))

## 5. Is the gap real?Both models scored the *same* test examples, so the predictions are paired and the right test is**McNemar's exact test**, not two independent proportion tests. Only the discordant pairs carryinformation about which model is better, so the effective sample size is the number of disagreements —not the size of the test set.

In [ ]:
from metrics.significance import mcnemar_test, significance_sentence

mc = mcnemar_test(y_test, rob_pred, base_pred, exact=True)
print("2x2 discordance table:", mc.table())
print("discordant pairs:", mc.n_discordant, "| exact p =", f"{mc.p_value:.4g}")
print()
print(significance_sentence("RoBERTa (fine-tuned)", "TF-IDF + LogReg", rob_ci, base_ci, mc))

## 6. Interpretability`gradient_saliency`, **not** "Grad-CAM" — it computes `‖∂logit/∂embedding‖₂`, which is gradient-normsaliency. Grad-CAM pools gradients per channel and weights the *activations* of a chosen layer.Different method, different guarantees.The gradient is taken w.r.t. the **word** embeddings only. The original notebook fed the output of thefull embedding module back in as `inputs_embeds`, which re-applied position embeddings, token-typeembeddings and LayerNorm a second time — so every attribution it produced was computed on an input themodel had never seen in training.

In [ ]:
import torch

from interpretability.saliency import gradient_saliency, word_embeddings_of

sample_ids = roberta.tokenizer(x_test[0], max_length=32, truncation=True,
                               padding=False, return_tensors="pt")["input_ids"].to(device)
mask = torch.ones_like(sample_ids)
roberta.model.eval()
with torch.no_grad():
    from_ids = roberta.model(input_ids=sample_ids, attention_mask=mask).logits
    from_words = roberta.model(inputs_embeds=word_embeddings_of(roberta.model, sample_ids),
                               attention_mask=mask).logits
    from_full = roberta.model(inputs_embeds=roberta.model.roberta.embeddings(input_ids=sample_ids),
                              attention_mask=mask).logits
print("FIXED  path max|Δlogit| vs input_ids:", f"{(from_ids - from_words).abs().max().item():.2e}")
print("BUGGY  path max|Δlogit| vs input_ids:", f"{(from_ids - from_full).abs().max().item():.2e}")

In [ ]:
attr = gradient_saliency(roberta.model, roberta.tokenizer, x_test[0],
                         max_len=cfg.MODEL.MAX_LEN, device=device)
print("predicted:", ["negative", "positive"][attr.predicted_label],
      "| true:", ["negative", "positive"][y_test[0]])
for token, score in attr.top(10):
    print(f"  {token.lstrip('Ġ'):<18} {score:.4f}")

In [ ]:
from interpretability.attention import last_layer_attention

amap = last_layer_attention(roberta.model, roberta.tokenizer, x_test[0],
                            max_len=cfg.MODEL.MAX_LEN, device=device)
print(f"layer {amap.layer}, mean over {amap.n_heads} heads, {len(amap.tokens)} inner tokens")
for token, weight in amap.most_attended(k=10):
    print(f"  {token.lstrip('Ġ'):<18} {weight:.4f}")

## 7. What this notebook is notThese numbers come from **`cfg/dev.yaml`** — a small calibration config. The published results, theirWilson intervals, the McNemar test and the four-cell preprocessing ablation all come from`cfg/small.yaml` and live in [`../reports/RESULTS.md`](../reports/RESULTS.md).Neither config is the notebook's original 5-epoch run: that is `cfg/default.yaml`, which was **notrun** because it exceeds this repo's 45-minute wall-clock cap. See[`../docs/adr/0004-subset-size-and-published-config.md`](../docs/adr/0004-subset-size-and-published-config.md).Reproduce the published numbers with:```bashmake data && make small && make report && make figures```